# UTAU Auto OTO - Colab Preprocess Only

This notebook is for running only the mel-coupled preprocessing pipeline on Google Colab.

Included:
- project snapshot sync
- Python and MFA install
- optional `stage_sources.py`
- `prepare_pairs.py` execution
- `prepared_auto_pairs.json` check
- copy prepared `dataset` back to Google Drive

Notes:
- Colab is Linux, so Korean tokenizer uses `python-mecab-ko`, not `eunjeon`.
- MFA is called by absolute binary path.
- Colab runtime is ephemeral, so copy results back to Drive at the end.


## 1. Mount Drive and set paths

Adjust these variables first.

- `USE_DRIVE_REPO_SNAPSHOT=True`: use an existing project snapshot from Drive
- `PROJECT_GIT_URL`: git clone URL if no Drive snapshot is used
- `STAGE_FROM_SOURCE=True`: rebuild `dataset` from original voicebank roots
- `STAGE_FROM_SOURCE=False`: use existing `dataset` inside the project


In [1]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

USE_DRIVE_REPO_SNAPSHOT = True
PROJECT_GIT_URL = 'https://github.com/SODAsoo07/Auto_OTO.git'
PROJECT_BRANCH = 'Mel-oto'
DRIVE_REPO_SNAPSHOT = Path('/content/drive/MyDrive/UTAU_Auto_OTO_v3/Auto_OTO')

WORK_ROOT = Path('/content/utoa_colab')
PROJECT_ROOT = WORK_ROOT / 'Auto_OTO'
DATASET_ROOT = PROJECT_ROOT / 'dataset'

STAGE_FROM_SOURCE = False
COLAB_TRAINING_ROOTS_YAML = PROJECT_ROOT / 'ml' / 'configs' / 'training_data_roots.colab.yaml'
VOICEBANK_ROOTS = {'japanese': {'cv': [], 'vcv': [], 'cvvc': []}, 'korean': {'cv': [], 'cvc': [], 'cvvc': [], 'vcv': []}}

LANGUAGES_TO_PREPARE = ['korean']
PREPARE_DRY_RUN = False
PREPARE_LIMIT = 0
PREPARE_RESUME = True
PREPARE_RETRY_FAILED = True

COPY_RESULTS_BACK_TO_DRIVE = True
DRIVE_RESULT_DATASET = Path('/content/drive/MyDrive/UTAU_Auto_OTO_v3/dataset_prepared')

WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f'WORK_ROOT={WORK_ROOT}')
print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'DATASET_ROOT={DATASET_ROOT}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
WORK_ROOT=/content/utoa_colab
PROJECT_ROOT=/content/utoa_colab/Auto_OTO
DATASET_ROOT=/content/utoa_colab/Auto_OTO/dataset


## 2. Helpers


In [2]:
import subprocess
import shutil

def run_cmd(args, env=None, cwd=None):
    printable = ' '.join(str(a) for a in args)
    print(f'$ {printable}')
    subprocess.run([str(a) for a in args], check=True, env=env, cwd=cwd)

def run_bash(cmd: str, env=None, cwd=None):
    print(f'$ {cmd}')
    subprocess.run(cmd, shell=True, check=True, executable='/bin/bash', env=env, cwd=cwd)


## 3. Sync project snapshot


In [3]:
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

if USE_DRIVE_REPO_SNAPSHOT:
    if not DRIVE_REPO_SNAPSHOT.exists():
        raise FileNotFoundError(f'Drive snapshot not found: {DRIVE_REPO_SNAPSHOT}')
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(['rsync', '-a', '--delete', '--exclude', '.git', f'{DRIVE_REPO_SNAPSHOT}/', f'{PROJECT_ROOT}/'])
else:
    if not PROJECT_GIT_URL.strip():
        raise ValueError('PROJECT_GIT_URL is required when USE_DRIVE_REPO_SNAPSHOT is False')
    run_cmd(['git', 'clone', '--depth', '1', '--branch', PROJECT_BRANCH, PROJECT_GIT_URL, str(PROJECT_ROOT)])

os.chdir(PROJECT_ROOT)
print('cwd =', Path.cwd())
print('dataset exists =', DATASET_ROOT.exists())


$ rsync -a --delete --exclude .git /content/drive/MyDrive/UTAU_Auto_OTO_v3/Auto_OTO/ /content/utoa_colab/Auto_OTO/
cwd = /content/utoa_colab/Auto_OTO
dataset exists = False


## 4. Install Python dependencies and MFA

Install policy:
- MFA `2.2.17`
- Python `3.10`
- Korean tokenizer: `python-mecab-ko`, `jamo`
- Japanese tokenizer: `spacy`, `sudachipy`, `sudachidict-core`

Important changes:
- `mfa version` check was removed.
- MFA is verified by absolute binary path using `--help`.
- `MFA_ROOT_DIR` is fixed to `/content/mfa_root`.


In [4]:
import os

MFA_VERSION = '2.2.17'
MFA_PYTHON = '3.10'
MAMBA_ROOT = Path('/content/micromamba')
MICROMAMBA_EXE = Path('/content/bin/micromamba')
MFA_ENV_PREFIX = MAMBA_ROOT / 'envs' / 'mfa'
MFA_BIN = MFA_ENV_PREFIX / 'bin' / 'mfa'
MFA_PY_BIN = MFA_ENV_PREFIX / 'bin' / 'python'
MFA_ROOT_DIR = Path('/content/mfa_root')
MFA_ROOT_DIR.mkdir(parents=True, exist_ok=True)

run_cmd(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run_cmd(['python', '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt'), '-r', str(PROJECT_ROOT / 'requirements-ml.txt'), 'pyyaml'])

if not MICROMAMBA_EXE.exists():
    Path('/content/bin').mkdir(parents=True, exist_ok=True)
    run_cmd(['wget', '-q', '-O', '/tmp/micromamba.tar.bz2', 'https://micro.mamba.pm/api/micromamba/linux-64/latest'])
    run_cmd(['tar', '-xjf', '/tmp/micromamba.tar.bz2', '-C', '/content', 'bin/micromamba'])

if not MFA_BIN.exists():
    run_cmd([str(MICROMAMBA_EXE), 'create', '-y', '-r', str(MAMBA_ROOT), '-p', str(MFA_ENV_PREFIX), '-c', 'conda-forge', f'python={MFA_PYTHON}', f'montreal-forced-aligner={MFA_VERSION}', 'openfst=1.8.2', 'kaldi=5.5.1068'], env={**os.environ, 'MAMBA_ROOT_PREFIX': str(MAMBA_ROOT)})

# Repair critical Python package compatibility inside MFA env.
run_cmd([str(MFA_PY_BIN), '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools==80.9.0', 'wheel', 'joblib>=1.3,<1.5', 'python-mecab-ko', 'jamo', 'spacy', 'sudachipy', 'sudachidict-core'])

os.environ['PATH'] = f"{MFA_ENV_PREFIX / 'bin'}:{os.environ['PATH']}"
os.environ['PYTHONUTF8'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['MFA_ROOT_DIR'] = str(MFA_ROOT_DIR)
os.environ['MPLBACKEND'] = 'Agg'
os.environ.setdefault('UTOA_ML_PREPARE_MFA_PROFILE', 'default')

mfa_env = os.environ.copy()
mfa_env['MPLBACKEND'] = 'Agg'

print(f'MFA_BIN={MFA_BIN}')
print(f'MFA exists={MFA_BIN.exists()}')

run_cmd([str(MFA_PY_BIN), '-c', 'import setuptools, pkg_resources, joblib; print("setuptools OK"); print(joblib.__version__)'], env=mfa_env)

result = subprocess.run(
    [str(MFA_BIN), '--help'],
    env=mfa_env,
    capture_output=True,
    text=True,
    timeout=30,
)

print("returncode =", result.returncode)
print("stdout tail =")
print((result.stdout or "")[-1000:])
print("stderr tail =")
print((result.stderr or "")[-1000:])

if result.returncode != 0:
    raise RuntimeError('MFA binary check failed')

run_cmd([str(MFA_PY_BIN), '-c', 'import mecab, jamo, spacy, sudachipy, sudachidict_core; print("MFA tokenizer deps OK")'], env=mfa_env)


$ python -m pip install -q --upgrade pip setuptools wheel
$ python -m pip install -q -r /content/utoa_colab/Auto_OTO/requirements.txt -r /content/utoa_colab/Auto_OTO/requirements-ml.txt pyyaml
$ /content/micromamba/envs/mfa/bin/python -m pip install -q --upgrade pip setuptools==80.9.0 wheel joblib>=1.3,<1.5 python-mecab-ko jamo spacy sudachipy sudachidict-core
MFA_BIN=/content/micromamba/envs/mfa/bin/mfa
MFA exists=True
$ /content/micromamba/envs/mfa/bin/python -c import setuptools, pkg_resources, joblib; print("setuptools OK"); print(joblib.__version__)
returncode = 0
stdout tail =
tabase servers            │
│ tokenize             Tokenize utterances                                     │
│ train                Train a new acoustic model                              │
│ train_dictionary     Calculate pronunciation probabilities                   │
│ train_g2p            Train a G2P model                                       │
│ train_ivector        Train an ivector extractor         

## 5. Download acoustic models


In [5]:
requested_models = []
if 'korean' in LANGUAGES_TO_PREPARE:
    requested_models.append('korean_mfa')
if 'japanese' in LANGUAGES_TO_PREPARE:
    requested_models.append('japanese_mfa')

for model_name in requested_models:
    run_cmd([str(MFA_BIN), 'model', 'download', 'acoustic', model_name, '--ignore_cache'], env=mfa_env)


$ /content/micromamba/envs/mfa/bin/mfa model download acoustic korean_mfa --ignore_cache


## 6. Optional: rebuild dataset from voicebank roots


In [7]:
!7za x /content/drive/MyDrive/archive/Voice_Datas.zip -o/content/utoa_colab/Auto_OTO/dataset

         45% 12143 - Kr/CVVC/Achu_CVVC/_feo'feo'fyeo'-'fyeo'fweo.w                                                           45% 121         45% 12200 - Kr/CVVC/Achu_CVVC/_la'l'la'l'lya'l'lwa.wa                                                       45% 12229 - Kr/CVVC/Achu_CVVC/_n'ya'n'ye'n'yo.w                                                 45% 12259 - Kr/CVVC/Achu_CVVC/_ppwa'-'ppwe'-'ppwi'-'ppweo.w                                                             46% 12287 - Kr/CVVC/Achu_CVVC/_te'te'tye'-'tye'twe.wa                                                       46% 12316 - Kr/CVVC/Achu_CVVC/_vu'vu'vyu'-'veui'veui.wa                                                         46% 12345 - Kr/CVVC/Dam_Kor_CVVC/_a'hwa'kkwa'ttwa'ppwa'sswa'jjwa.wa                                                                     46% 12367 - Kr/CVVC/Dam_Kor_CVVC/_eo'hweo'kkweo'ttweo'ppweo'ssweo'jjweo.w                                                                           46% 12395 - Kr/CVVC/Dam_Kor_CVV

In [9]:
import yaml

if STAGE_FROM_SOURCE:
    config_payload = {'japanese': VOICEBANK_ROOTS.get('japanese', {}), 'korean': VOICEBANK_ROOTS.get('korean', {}), 'notes': {'recursive_oto_search': True, 'recursive_wav_search': True, 'strip_pitch_suffix_for_matching': True}}
    COLAB_TRAINING_ROOTS_YAML.parent.mkdir(parents=True, exist_ok=True)
    with open(COLAB_TRAINING_ROOTS_YAML, 'w', encoding='utf-8') as f:
        yaml.safe_dump(config_payload, f, allow_unicode=True, sort_keys=False)
    run_cmd(['python', str(PROJECT_ROOT / 'ml' / 'scripts' / 'coupled' / 'stage_sources.py'), '--config', str(COLAB_TRAINING_ROOTS_YAML), '--dataset-root', str(DATASET_ROOT)], env=mfa_env)
else:
    print('STAGE_FROM_SOURCE=False: skip stage_sources.py')
    if not DATASET_ROOT.exists():
        raise FileNotFoundError(f'dataset folder not found: {DATASET_ROOT}')


STAGE_FROM_SOURCE=False: skip stage_sources.py


## 7. Run prepare_pairs.py

Current pipeline behavior:
- `--resume` checkpoint resume
- `--retry-failed` retry failed items
- Korean uses `default -> accurate -> fast` fallback
- alias normalization uses `prefix.map` first, then parseable `*.map` fallback
- breath aliases are normalized to `br`


In [17]:
PREPARE_RESUME = False
PREPARE_RETRY_FAILED = False


In [14]:
wav_count = sum(1 for _ in DATASET_ROOT.rglob('*.wav'))
oto_count = sum(1 for _ in DATASET_ROOT.rglob('oto.ini'))
manifest_path = DATASET_ROOT / '_manifest' / 'prepared_auto_pairs.json'

print('DATASET_ROOT =', DATASET_ROOT)
print('wav_count =', wav_count)
print('oto_count =', oto_count)
print('manifest_exists =', manifest_path.exists())


DATASET_ROOT = /content/utoa_colab/Auto_OTO/dataset
wav_count = 21894
oto_count = 158
manifest_exists = True


In [18]:
prepare_cmd = [
    'python',
    '-u',
    str(PROJECT_ROOT / 'ml' / 'scripts' / 'coupled' / 'prepare_pairs.py'),
    '--dataset-root',
    str(DATASET_ROOT),
]

if PREPARE_DRY_RUN:
    prepare_cmd.append('--dry-run')
if PREPARE_LIMIT and int(PREPARE_LIMIT) > 0:
    prepare_cmd.extend(['--limit', str(int(PREPARE_LIMIT))])
if PREPARE_RESUME:
    prepare_cmd.append('--resume')
if PREPARE_RETRY_FAILED:
    prepare_cmd.append('--retry-failed')

print('RUN =', ' '.join(prepare_cmd))

result = subprocess.run(
    prepare_cmd,
    env=mfa_env,
    capture_output=True,
    text=True,
)

print('returncode =', result.returncode)
print('stdout =')
print(result.stdout)
print('stderr =')
print(result.stderr)


RUN = python -u /content/utoa_colab/Auto_OTO/ml/scripts/coupled/prepare_pairs.py --dataset-root /content/utoa_colab/Auto_OTO/dataset
returncode = 0
stdout =
[Prepare] 시작: total=0 dry_run=False mfa=OK resume=False
[Prepare] 종료: prepared=0 skipped=0 pending=0 total=0
{
  "summary": {
    "total_items": 0,
    "prepared": 0,
    "skipped": 0,
    "dry_run_items": 0,
    "pending": 0,
    "dry_run": false,
    "mfa_path": "/content/micromamba/envs/mfa/bin/mfa",
    "resume": false,
    "retry_failed": false,
    "resume_skipped": 0
  },
  "report_path": "/content/utoa_colab/Auto_OTO/dataset/_manifest/prepared_auto_pairs.json"
}

stderr =



In [22]:
from pathlib import Path

print('dataset children:')
for p in sorted(DATASET_ROOT.iterdir()):
    print('-', p.name)

print('\nexists korean =', (DATASET_ROOT / 'korean').exists())
print('exists japanese =', (DATASET_ROOT / 'japanese').exists())


dataset children:
- Jp
- Kr
- _manifest
- 타이밍 보정 모델 보이스뱅크 데이터 제공자.txt

exists korean = False
exists japanese = False


In [25]:
from pathlib import Path
import shutil

jp_src = DATASET_ROOT / 'Jp'
kr_src = DATASET_ROOT / 'Kr'
jp_dst = DATASET_ROOT / 'japanese'
kr_dst = DATASET_ROOT / 'korean'

if kr_src.exists() and not kr_dst.exists():
    print(f'rename: {kr_src} -> {kr_dst}')
    shutil.move(str(kr_src), str(kr_dst))

if jp_src.exists() and not jp_dst.exists():
    print(f'rename: {jp_src} -> {jp_dst}')
    shutil.move(str(jp_src), str(jp_dst))

print('children after rename:')
for p in sorted(DATASET_ROOT.iterdir()):
    print('-', p.name)


rename: /content/utoa_colab/Auto_OTO/dataset/Kr -> /content/utoa_colab/Auto_OTO/dataset/korean
rename: /content/utoa_colab/Auto_OTO/dataset/Jp -> /content/utoa_colab/Auto_OTO/dataset/japanese
children after rename:
- _manifest
- japanese
- korean
- 타이밍 보정 모델 보이스뱅크 데이터 제공자.txt


In [26]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from core.oto_ml_prepare_discovery import _discover_work_items

items = _discover_work_items(str(DATASET_ROOT))
print('discovered =', len(items))
for item in items[:10]:
    print(item.language, item.format_type, item.work_dir)


discovered = 155
korean VCV /content/utoa_colab/Auto_OTO/dataset/korean/VCV/MRU-U ko_VCV
korean VCV /content/utoa_colab/Auto_OTO/dataset/korean/VCV/JN_KR_VCV/JYF3
korean VCV /content/utoa_colab/Auto_OTO/dataset/korean/VCV/JN_KR_VCV/JYFA
korean VCV /content/utoa_colab/Auto_OTO/dataset/korean/VCV/JN_KR_VCV/JYG#4
korean VCV /content/utoa_colab/Auto_OTO/dataset/korean/VCV/JN_KR_VCV/JYC4
korean VCV /content/utoa_colab/Auto_OTO/dataset/korean/VCV/DD_VCV
korean VCV /content/utoa_colab/Auto_OTO/dataset/korean/VCV/MmKR_VCV
korean VCV /content/utoa_colab/Auto_OTO/dataset/korean/VCV/Song-SoRi_Double Diamond KO_VCV
korean VCV /content/utoa_colab/Auto_OTO/dataset/korean/VCV/WinnieKRVCV
korean VCV /content/utoa_colab/Auto_OTO/dataset/korean/VCV/Jan_VCV


In [ ]:
prepare_cmd = ['python', str(PROJECT_ROOT / 'ml' / 'scripts' / 'coupled' / 'prepare_pairs.py'), '--dataset-root', str(DATASET_ROOT)]
PREPARE_RESUME = False
PREPARE_RETRY_FAILED = False
if PREPARE_DRY_RUN:
    prepare_cmd.append('--dry-run')
if PREPARE_LIMIT and int(PREPARE_LIMIT) > 0:
    prepare_cmd.extend(['--limit', str(int(PREPARE_LIMIT))])
if PREPARE_RESUME:
    prepare_cmd.append('--resume')
if PREPARE_RETRY_FAILED:
    prepare_cmd.append('--retry-failed')
run_cmd(prepare_cmd, env=mfa_env)


$ python /content/utoa_colab/Auto_OTO/ml/scripts/coupled/prepare_pairs.py --dataset-root /content/utoa_colab/Auto_OTO/dataset


## 8. Inspect prepare report


In [11]:
import json
import pandas as pd

report_path = DATASET_ROOT / '_manifest' / 'prepared_auto_pairs.json'
if not report_path.exists():
    raise FileNotFoundError(f'prepare report not found: {report_path}')
with open(report_path, 'r', encoding='utf-8') as f:
    payload = json.load(f)
summary = payload.get('summary', {})
items = payload.get('items', [])
print(json.dumps(summary, ensure_ascii=False, indent=2))
df = pd.DataFrame(items)
if not df.empty:
    cols = [c for c in ['language', 'format_type', 'work_dir', 'status', 'reason'] if c in df.columns]
    display(df[cols].head(20))
    failed_df = df[df['status'] == 'skip'][cols] if 'status' in df.columns else pd.DataFrame()
    print(f'failed_count={len(failed_df)}')
    if len(failed_df):
        display(failed_df.head(50))


{
  "total_items": 0,
  "prepared": 0,
  "skipped": 0,
  "dry_run_items": 0,
  "pending": 0,
  "dry_run": false,
  "mfa_path": "/content/micromamba/envs/mfa/bin/mfa",
  "resume": true,
  "retry_failed": true,
  "resume_skipped": 0
}


## 9. Copy prepared dataset back to Drive


In [ ]:
if COPY_RESULTS_BACK_TO_DRIVE:
    DRIVE_RESULT_DATASET.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(['rsync', '-a', '--delete', f'{DATASET_ROOT}/', f'{DRIVE_RESULT_DATASET}/'])
    print(f'copied_to={DRIVE_RESULT_DATASET}')
else:
    print('COPY_RESULTS_BACK_TO_DRIVE=False: skip copy back')


## 10. Optional: delete generated alignment artifacts only


In [ ]:
# cleanup_cmd = "find '{root}' -type f \( -name '*.lab' -o -name '*.TextGrid' -o -name 'dictionary_auto.txt' -o -name 'oto_auto_ml.ini' \) -delete".format(root=DATASET_ROOT)
# run_bash(cleanup_cmd)
# print('generated alignment artifacts removed')
